In [5]:
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

True

In [9]:
import os
import json
import time
import docx
from pypdf import PdfReader
from dotenv import load_dotenv
from google import genai
from google.genai import types
from google.genai.errors import APIError

# --- Load Environment Variables ---
load_dotenv()

# --- Configuration ---
RAW_DATA_DIR = "."  # Current directory containing PDFs and DOCX files
GENERATION_OUTPUT_FILE = "generation_data.jsonl"
GRADING_OUTPUT_FILE = "grading_data.jsonl"

# Initialize Google GenAI client (uses GEMINI_API_KEY from environment/.env)
client = genai.Client()


def extract_text_from_files(data_dir: str) -> str:
    """Extracts raw text from all PDF and DOCX files in the specified directory."""
    combined_text = ""
    for filename in os.listdir(data_dir):
        filepath = os.path.join(data_dir, filename)
        
        # Skip temporary Office files
        if filename.startswith("~$"):
            continue

        # Extract PDF
        if filename.endswith(".pdf"):
            print(f"📄 Extracting PDF: {filename}")
            try:
                reader = PdfReader(filepath)
                for page in reader.pages:
                    text = page.extract_text()
                    if text:
                        combined_text += text + "\n"
            except Exception as e:
                print(f"⚠️ Error reading PDF {filename}: {e}")
                    
        # Extract Word Document (.docx)
        elif filename.endswith(".docx"):
            print(f"📝 Extracting Word Doc: {filename}")
            try:
                doc = docx.Document(filepath)
                for para in doc.paragraphs:
                    if para.text.strip():
                        combined_text += para.text + "\n"
            except Exception as e:
                print(f"⚠️ Error reading DOCX {filename}: {e}")

    return combined_text


def generate_training_pairs(context_chunk: str, dataset_type: str, max_retries: int = 5) -> list[dict]:
    """Uses Gemini API to synthesize structured instruction-output pairs with retry logic on errors."""
    
    if dataset_type == "generation":
        system_instruction = (
            "You are an expert engineering curriculum designer. "
            "Extract topics and synthesize question-generation pairs based on the provided course material. "
            "Output MUST be a JSON list of objects, where each object has 'instruction' and 'output' fields."
        )
        prompt = f"""
Given the following syllabus/course material:
---
{context_chunk}
---

Generate 3 diverse fine-tuning examples for an AI question-generation model.
The 'instruction' must contain the target Course Outcome (CO), Bloom's Taxonomy level, marks, and topic context.
The 'output' must be a valid, stringified JSON object containing:
- 'question': Clear, academic question text
- 'co_po_mapping': e.g., {{"CO": "CO1", "PO": ["PO1", "PO2"]}}
- 'marks': Total marks (e.g., 5, 10)
- 'rubric': Detailed marking criteria breakdown

Respond ONLY with a JSON array formatted like:
[
  {{
    "instruction": "Generate a question for CO2 at Bloom's level 'Apply' worth 10 marks based on the following context: ...",
    "output": "{{\\\"question\\\": \\\"...\\\", \\\"co_po_mapping\\\": {{...}}, \\\"marks\\\": 10, \\\"rubric\\\": {{...}}}}"
  }}
]
"""
    else:  # grading
        system_instruction = (
            "You are an engineering course evaluator. "
            "Synthesize automated grading pairs from the provided questions and solutions. "
            "Output MUST be a JSON list of objects with 'instruction' and 'output' fields."
        )
        prompt = f"""
Given the following exam/solution context:
---
{context_chunk}
---

Generate 3 diverse fine-tuning examples for an AI automated grading model.
The 'instruction' must include the question, rubric/solution, and a student's answer (some completely correct, some with minor errors, some incorrect).
The 'output' must be a valid, stringified JSON object containing:
- 'score': Numeric score awarded
- 'feedback': Detailed constructive feedback
- 'rubric_breakdown': Itemized score breakdown corresponding to rubric items

Respond ONLY with a JSON array formatted like:
[
  {{
    "instruction": "Grade the following student answer... \\n**Question:** ... \\n**Rubric:** ... \\n**Student Answer:** ...",
    "output": "{{\\\"score\\\": 8, \\\"feedback\\\": \\\"...\\\", \\\"rubric_breakdown\\\": {{...}}}}"
  }}
]
"""

    for attempt in range(1, max_retries + 1):
        try:
            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=system_instruction,
                    response_mime_type="application/json",
                    temperature=0.3,
                ),
            )

            data = json.loads(response.text)
            time.sleep(1)  # Small pacing delay to respect API rates
            return data if isinstance(data, list) else []

        except Exception as e:
            wait_time = attempt * 5  # Delays: 5s, 10s, 15s, 20s, 25s
            print(f"⚠️ API error on attempt {attempt}/{max_retries}: {e}")
            if attempt < max_retries:
                print(f"⏳ Retrying in {wait_time} seconds...")
                time.sleep(wait_time)
            else:
                print("❌ Max retries reached for this chunk. Skipping...")
                return []


def format_to_chatml(instruction: str, output: str) -> dict:
    """Wraps raw instruction-output pairs into ChatML structure expected by Qwen."""
    return {
        "text": f"<|im_start|>user\n{instruction}<|im_end|>\n<|im_start|>assistant\n{output}<|im_end|>"
    }


def process_and_save(data_dir: str):
    if not os.path.exists(data_dir):
        print(f"❌ Directory '{data_dir}' not found.")
        return

    raw_text = extract_text_from_files(data_dir)
    if not raw_text.strip():
        print("❌ No text extracted from directory.")
        return

    # Chunk text into ~2000-character blocks
    chunks = [raw_text[i:i + 2000] for i in range(0, len(raw_text), 1800)]
    print(f"🧩 Split extracted text into {len(chunks)} chunks for batch processing.")

    generation_entries = []
    grading_entries = []

    # Process up to the first 10 chunks as an initial batch
    total_chunks_to_process = min(10, len(chunks))
    for idx, chunk in enumerate(chunks[:total_chunks_to_process]):
        print(f"\n🔄 Processing chunk {idx + 1}/{total_chunks_to_process}...")
        
        # Synthesize Generation Pairs
        gen_pairs = generate_training_pairs(chunk, dataset_type="generation")
        for pair in gen_pairs:
            if "instruction" in pair and "output" in pair:
                generation_entries.append(format_to_chatml(pair["instruction"], pair["output"]))

        # Synthesize Grading Pairs
        grad_pairs = generate_training_pairs(chunk, dataset_type="grading")
        for pair in grad_pairs:
            if "instruction" in pair and "output" in pair:
                grading_entries.append(format_to_chatml(pair["instruction"], pair["output"]))

    # Write Generation Dataset
    with open(GENERATION_OUTPUT_FILE, "w", encoding="utf-8") as f:
        for entry in generation_entries:
            f.write(json.dumps(entry) + "\n")
    print(f"✅ Saved {len(generation_entries)} entries to {GENERATION_OUTPUT_FILE}")

    # Write Grading Dataset
    with open(GRADING_OUTPUT_FILE, "w", encoding="utf-8") as f:
        for entry in grading_entries:
            f.write(json.dumps(entry) + "\n")
    print(f"✅ Saved {len(grading_entries)} entries to {GRADING_OUTPUT_FILE}")


if __name__ == "__main__":
    process_and_save(RAW_DATA_DIR)

📄 Extracting PDF: AAP _Odd Sem_5_ 2026-2027_ML_KGD.pdf
📄 Extracting PDF: AAP-PCD2-RVD-Odd-2026-27.pdf
📄 Extracting PDF: AAP_AI – Semester V- INFT - SM.pdf
📄 Extracting PDF: AAP_V_INFT_2026-27_NSM.pdf
📄 Extracting PDF: CG AAP odd 26-27_Final.pdf
📄 Extracting PDF: CG-MSE_I-Solution.pdf
📄 Extracting PDF: CG-MSE_II-Solution.pdf
📄 Extracting PDF: cgsol25_26.pdf
📝 Extracting Word Doc: CG_Assignment10_Advanced_Rendering_Solution.docx
📝 Extracting Word Doc: CG_Assignment11_Poster_Solution.docx
📄 Extracting PDF: CG_Assignment11_Poster_Virtual_Production.pdf
📝 Extracting Word Doc: CG_Assignment1_Design_Analysis_Solution.docx
📝 Extracting Word Doc: CG_Assignment2_Graphics_Algorithms_Solution.docx
📝 Extracting Word Doc: CG_Assignment3_VR_AR_Prototype_Solution.docx
📝 Extracting Word Doc: CG_Assignment4_Map_Visualization_Transformations_Solution.docx
📝 Extracting Word Doc: CG_Assignment5_Clipping_Application_Solution.docx
📝 Extracting Word Doc: CG_Assignment6_Texture_Mapping_Summary_Solution.docx
📝 